# model

LeNet-5 (LeCun et al., 1998) — phiên bản hiện đại dùng ReLU + MaxPool.

conv1 5x5, 6 kênh (padding 2) -> ReLU -> MaxPool 2
conv2 5x5, 16 kênh            -> ReLU -> MaxPool 2
fc1 400->120 -> ReLU -> fc2 120->84 -> ReLU -> fc3 84->10
Tổng 61 706 tham số.

## Mục tiêu và cách đọc

Đây là thư viện hỗ trợ, không phải thí nghiệm độc lập. Các notebook 01–11 tự nhập thư viện này. Bật RUN_EXPERIMENT để nạp các định nghĩa khi muốn tìm hiểu.

## Cách sử dụng

1. Chọn kernel Python có `ipykernel`, `numpy`, `torch`, `matplotlib`.
2. Giữ `RUN_EXPERIMENT = False` và chọn **Run All** để xem dữ liệu sẵn có.
3. Đổi thành `True` nếu muốn thực hiện lại; chạy tuần tự các ô từ đầu.
4. **Ctrl+S / Cmd+S** lưu toàn bộ output vào notebook. Báo cáo HTML và CSV được xuất tự động ở ô cuối.

**Lưu ý:** các hình B/C/D là file đã lưu; sau khi đổi kết quả, chạy notebook 08 để vẽ lại trước khi dùng hình trong bài. Thời điểm xuất báo cáo không phải thời điểm thực hiện thí nghiệm gốc.


## 1. Thiết lập và chọn chế độ

Cài môi trường một lần theo `HUONG_DAN_NOTEBOOK.md`. Mỗi lần chạy tạo thư mục riêng trong `results/notebook_runs/`. Khi chạy lại thí nghiệm, các kết quả chuẩn và checkpoint hiện có được sao lưu trước.

In [ ]:
from pathlib import Path
import os
import sys

# Tìm 02_demo dù kernel bắt đầu ở thư mục workspace hoặc thư mục notebook.
candidates = [Path.cwd(), *Path.cwd().parents]
DEMO_ROOT = next((base / suffix for base in candidates
                  for suffix in ("", "02_demo", "Gk_XLDL/02_demo")
                  if (base / suffix / "posit_lab/numformats.py").is_file()), None)
if DEMO_ROOT is None:
    raise FileNotFoundError("Mở notebook trong workspace chứa Gk_XLDL/02_demo.")
DEMO_ROOT = DEMO_ROOT.resolve()
os.chdir(DEMO_ROOT)
if str(DEMO_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMO_ROOT))
from notebook_tools.reporting import NotebookRun

# False: xem số liệu sẵn có. True: chạy code thí nghiệm bên dưới.
RUN_EXPERIMENT = False
SCRIPT_ARGS = []
__file__ = str(DEMO_ROOT / 'posit_lab/model.py')
__package__ = 'posit_lab'
session = NotebookRun(DEMO_ROOT, 'posit_lab/model.py', RUN_EXPERIMENT, SCRIPT_ARGS)
print("Thư mục lưu lần này:", session.out)

# Cell magic giữ code thẳng hàng; vẫn áp dụng chế độ chạy và lưu nhật ký.
from IPython.core.magic import register_cell_magic
import __future__

@register_cell_magic
def experiment(line, cell):
    if not RUN_EXPERIMENT:
        print("Đã bỏ qua tính toán; đặt RUN_EXPERIMENT = True để chạy ô này.")
        return
    with session.execute():
        code = compile(cell, str(session.source), "exec",
                       flags=__future__.annotations.compiler_flag)
        exec(code, get_ipython().user_ns)


## 2. LeNet5, evaluate

Mã chuyển nguyên từ `posit_lab/model.py`. Ô chỉ thực thi khi `RUN_EXPERIMENT = True`; giữ thứ tự để các biến và hàm được khởi tạo đúng.

In [ ]:
%%experiment
"""LeNet-5 (LeCun et al., 1998) — phiên bản hiện đại dùng ReLU + MaxPool.

conv1 5x5, 6 kênh (padding 2) -> ReLU -> MaxPool 2
conv2 5x5, 16 kênh            -> ReLU -> MaxPool 2
fc1 400->120 -> ReLU -> fc2 120->84 -> ReLU -> fc3 84->10
Tổng 61 706 tham số.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


class LeNet5(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


LAYERS = ["conv1", "conv2", "fc1", "fc2", "fc3"]


@torch.no_grad()
def evaluate(model, x, y, batch=2000):
    model.eval()
    correct = 0
    for i in range(0, len(x), batch):
        correct += (model(x[i:i + batch]).argmax(1) == y[i:i + batch]).sum().item()
    return 100.0 * correct / len(x)


## Kết quả, bảng số liệu và hình

Ô cuối đọc đầy đủ các trường JSON, xuất CSV UTF-8 cho Excel, lưu JSON gốc, nhúng hình vào HTML để mở độc lập. Các bảng dài có thanh cuộn. Khi viết bài, ghi rõ cấu hình, seed, tập dữ liệu và giới hạn mô phỏng.

In [ ]:
session.report([], [])